# 🌳 Ejercicios de Árboles Binarios

Este notebook contiene tres ejercicios sobre árboles binarios usando la estructura de datos ya definida en el repositorio.

---

## Estructura Base

A continuación se define la misma estructura `BinaryTreeNode` y la función `BST` del repositorio (`SingleCode/BST.py`), junto con un helper de visualización.

In [ ]:
class BinaryTreeNode:
    def __init__(self, value):
        self.value = value
        self.left = None
        self.right = None

    def visualize(self, level=0, prefix="Root: "):
        print(" " * (level * 4) + prefix + str(self.value))
        if self.left:
            self.left.visualize(level + 1, prefix="L--- ")
        if self.right:
            self.right.visualize(level + 1, prefix="R--- ")


def BST(nums: list[int], Left: int = 0, Right: int = None):
    """Construye un árbol binario de búsqueda balanceado a partir de una lista ordenada."""
    if Right is None:
        Right = len(nums) - 1

    if Left > Right:
        return None

    if Left == Right:
        return BinaryTreeNode(nums[Left])

    Mid = (Left + Right) // 2
    node = BinaryTreeNode(nums[Mid])
    node.left = BST(nums, Left, Mid - 1)
    node.right = BST(nums, Mid + 1, Right)
    return node


# Árbol de prueba: [-10, -3, 0, 5, 9]
nums = [-10, -3, 0, 5, 9]
root = BST(nums)
print("Árbol de prueba:")
root.visualize()

---
## Ejercicio 1 — Eliminación de Nodos en un BST

### Descripción
Dado un árbol binario de búsqueda (BST) y un valor `key`, elimina el nodo con ese valor y devuelve la raíz del árbol resultante. El árbol debe seguir siendo un BST válido después de la eliminación.

### Casos de prueba

| Árbol inicial (inorden) | key | Árbol resultante (inorden) |
|---|---|---|
| `[-10, -3, 0, 5, 9]` | `0` | `[-10, -3, 5, 9]` |
| `[-10, -3, 0, 5, 9]` | `-10` | `[-3, 0, 5, 9]` |
| `[-10, -3, 0, 5, 9]` | `9` | `[-10, -3, 0, 5]` |

### Estrategia
1. Si `key < node.value` → buscar en subárbol izquierdo.
2. Si `key > node.value` → buscar en subárbol derecho.
3. Si `key == node.value`:
   - Nodo hoja → devolver `None`.
   - Solo hijo derecho → devolver hijo derecho.
   - Solo hijo izquierdo → devolver hijo izquierdo.
   - Dos hijos → reemplazar con el **sucesor inorden** (mínimo del subárbol derecho) y eliminar ese sucesor del subárbol derecho.

In [ ]:
def inorder(node: BinaryTreeNode) -> list:
    """Recorrido inorden para verificar resultados."""
    if node is None:
        return []
    return inorder(node.left) + [node.value] + inorder(node.right)


def find_min(node: BinaryTreeNode) -> BinaryTreeNode:
    """Retorna el nodo con el valor mínimo en un subárbol."""
    while node.left is not None:
        node = node.left
    return node


def delete_node(root: BinaryTreeNode, key: int) -> BinaryTreeNode:
    """
    Elimina el nodo con valor `key` del BST.

    Parámetros:
        root: raíz del BST.
        key: valor del nodo a eliminar.

    Retorna:
        La nueva raíz del BST después de la eliminación.
    """
    if root is None:
        return None

    if key < root.value:
        root.left = delete_node(root.left, key)
    elif key > root.value:
        root.right = delete_node(root.right, key)
    else:
        # Caso 1: nodo hoja o solo un hijo
        if root.left is None:
            return root.right
        if root.right is None:
            return root.left

        # Caso 2: dos hijos → reemplazar con sucesor inorden
        successor = find_min(root.right)
        root.value = successor.value
        root.right = delete_node(root.right, successor.value)

    return root


# --- Pruebas ---
test_cases = [
    (BST([-10, -3, 0, 5, 9]),  0, [-10, -3, 5, 9]),
    (BST([-10, -3, 0, 5, 9]), -10, [-3, 0, 5, 9]),
    (BST([-10, -3, 0, 5, 9]),   9, [-10, -3, 0, 5]),
]

for tree, key, expected in test_cases:
    result = delete_node(tree, key)
    got = inorder(result)
    status = "✅" if got == expected else "❌"
    print(f"{status} delete({key}): esperado {expected}, obtenido {got}")

---
## Ejercicio 2 — Buscar Pares Padre-Hijo donde `child % father == 0`

### Descripción
Dado un árbol binario (no necesariamente BST), encontrar todos los pares `(padre, hijo)` tales que el valor del hijo sea divisible por el valor del padre (`child % father == 0`). Los nodos con valor `0` se ignoran como padre (división por cero).

### Casos de prueba

Árbol de ejemplo:
```
        3
       / \
      6   9
     / \
    12   2
```
Pares encontrados (DFS preorden): `(3, 6)`, `(6, 12)`, `(3, 9)`

### Estrategia
Recorrer el árbol con DFS pasando el valor del padre. En cada nodo, si `father != 0` y `node.value % father == 0`, agregar el par a la lista de resultados.

In [ ]:
def find_divisible_pairs(
    node: BinaryTreeNode,
    parent_value: int = None,
    result: list = None
) -> list[tuple]:
    """
    Encuentra todos los pares (padre, hijo) donde child % father == 0.

    Parámetros:
        node: nodo actual del árbol.
        parent_value: valor del nodo padre (None si es la raíz).
        result: lista acumuladora de pares encontrados.

    Retorna:
        Lista de tuplas (padre, hijo).
    """
    if result is None:
        result = []

    if node is None:
        return result

    if parent_value is not None and parent_value != 0:
        if node.value % parent_value == 0:
            result.append((parent_value, node.value))

    find_divisible_pairs(node.left,  node.value, result)
    find_divisible_pairs(node.right, node.value, result)

    return result


# --- Construcción manual del árbol de prueba ---
#        3
#       / \
#      6   9
#     / \
#    12   2
tree2 = BinaryTreeNode(3)
tree2.left = BinaryTreeNode(6)
tree2.right = BinaryTreeNode(9)
tree2.left.left = BinaryTreeNode(12)
tree2.left.right = BinaryTreeNode(2)

print("Árbol de prueba:")
tree2.visualize()

pairs = find_divisible_pairs(tree2)
print(f"\nPares (padre, hijo) donde child % father == 0: {pairs}")
assert sorted(pairs) == sorted([(3, 6), (6, 12), (3, 9)]), f"Resultado inesperado: {pairs}"
print("✅ Prueba 1 pasada")

# Prueba 2: árbol del BST original
#       0
#      / \
#    -3   5
#    /     \
#  -10      9
root2 = BST([-10, -3, 0, 5, 9])
pairs2 = find_divisible_pairs(root2)
print(f"Pares en BST original (child % father == 0): {pairs2}")

---
## Ejercicio 3 — LeetCode 112: Path Sum

**Fuente:** [LeetCode #112 – Path Sum](https://leetcode.com/problems/path-sum/)

### Descripción exacta del problema (traducción)
Dado la raíz de un árbol binario y un entero `targetSum`, devuelve `True` si el árbol tiene un camino **raíz a hoja** tal que la suma de todos los valores de los nodos del camino sea igual a `targetSum`.

Un **nodo hoja** es un nodo sin hijos.

### Entradas y salidas (3 ejemplos)

#### Entrada 1
```
       5
      / \
     4   8
    /   / \
   11  13   4
  /  \       \
 7    2       1
```
`targetSum = 22`  →  **Salida:** `True`  (camino: 5 → 4 → 11 → 2)

#### Entrada 2
```
    1
   / \
  2   3
```
`targetSum = 5`  →  **Salida:** `False`

#### Entrada 3
Árbol vacío, `targetSum = 0`  →  **Salida:** `False`

### Estrategia (DFS — paradigma de árbol)
Recorrer el árbol con DFS. En cada nodo restar su valor al `targetSum`. Al llegar a una hoja, verificar si el acumulado restante es `0`.

In [ ]:
def has_path_sum(root: BinaryTreeNode, target_sum: int) -> bool:
    """
    LeetCode 112 – Path Sum.

    Retorna True si existe un camino raíz-hoja cuya suma de valores
    sea igual a target_sum.

    Complejidad: O(n) tiempo, O(h) espacio (h = altura del árbol).
    """
    if root is None:
        return False

    remaining = target_sum - root.value

    # Nodo hoja: verificar si se completó el objetivo
    if root.left is None and root.right is None:
        return remaining == 0

    # DFS en ambos subárboles
    return (
        has_path_sum(root.left,  remaining) or
        has_path_sum(root.right, remaining)
    )


# --- Ejemplo 1 ---
#        5
#       / \
#      4   8
#     /   / \
#    11  13   4
#   /  \       \
#  7    2       1
t1 = BinaryTreeNode(5)
t1.left = BinaryTreeNode(4)
t1.right = BinaryTreeNode(8)
t1.left.left = BinaryTreeNode(11)
t1.left.left.left = BinaryTreeNode(7)
t1.left.left.right = BinaryTreeNode(2)
t1.right.left = BinaryTreeNode(13)
t1.right.right = BinaryTreeNode(4)
t1.right.right.right = BinaryTreeNode(1)

assert has_path_sum(t1, 22) == True,  "Error en ejemplo 1"
print("✅ Ejemplo 1: target=22 →", has_path_sum(t1, 22))

# --- Ejemplo 2 ---
#   1
#  / \
# 2   3
t2 = BinaryTreeNode(1)
t2.left = BinaryTreeNode(2)
t2.right = BinaryTreeNode(3)

assert has_path_sum(t2, 5) == False, "Error en ejemplo 2"
print("✅ Ejemplo 2: target=5  →", has_path_sum(t2, 5))

# --- Ejemplo 3: árbol vacío ---
assert has_path_sum(None, 0) == False, "Error en ejemplo 3"
print("✅ Ejemplo 3: vacío     →", has_path_sum(None, 0))

# --- Prueba extra con el BST del repositorio ---
root_bst = BST([-10, -3, 0, 5, 9])
print("\nBST original:")
root_bst.visualize()
# Camino: 0 → -3 → -10  suma = -13
print("\nhas_path_sum(BST, -13):", has_path_sum(root_bst, -13))
# Camino: 0 → 5 → 9  suma = 14
print("has_path_sum(BST,  14):", has_path_sum(root_bst, 14))